# 05 — DifIISR Target Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/05_difiisr_target_dynamics.ipynb)

Diagnostic only: **no guidance and no model modification**. We inspect target-local SCR and calibrated Hoyer in the frozen Original DifIISR one-step output.

We begin with a small deterministic sample: up to 15 validation targets from each LR size bin `≤2`, `2–4`, `4–8`. The frozen LR set, pinned DifIISR commit, checkpoint, and seed are the same as the baseline.


## 1. Setup

Run on a GPU runtime. This reuses the isolated DifIISR environment from the baseline design and does **not** regenerate LR.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,subprocess,json,textwrap
PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345; N_PER_BIN=15
DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
OUT=DATASET_ROOT/'difiisr_target_dynamics'; OUT.mkdir(parents=True,exist_ok=True)
project_dir=Path('/content/target-aware-infrared-sr'); difiisr_dir=Path('/content/DifIISR')
def run(cmd,cwd=None): print('$',' '.join(map(str,cmd))); subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)
if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir); run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)
ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv']); run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'pyiqa==0.1.12','basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown'])
weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True); ckpt=weights/'DifIISR.pth'
if not ckpt.exists(): run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',ckpt])
print('Frozen LR:',LR_DIR); print('Output:',OUT)


## 2. Select tiny targets

Selection uses the corrected validation YOLO labels and actual LR dimensions. The same image can contribute more than one target if applicable.


In [ ]:
import pandas as pd, numpy as np
ANN=project_dir/'annotations'/'corrected_yolo'/'valid'/'labels'
from PIL import Image
rows=[]
for lp in sorted(ANN.glob('*.txt')):
    ip=next(iter(LR_DIR.glob(lp.stem+'.*')),None)
    if ip is None: continue
    with Image.open(ip) as im: W,H=im.size
    for idx,line in enumerate([x for x in lp.read_text().splitlines() if x.strip()]):
        p=line.split();
        if len(p)<5: continue
        _,xc,yc,bw,bh=map(float,p[:5]); md=max(bw*W,bh*H)
        b='≤2 px' if md<=2 else '2–4 px' if md<=4 else '4–8 px' if md<=8 else None
        if b: rows.append(dict(image=ip.name,target_idx=idx,xc=xc,yc=yc,bw=bw,bh=bh,max_dim_lr=md,lr_size_bin=b))
targets=pd.DataFrame(rows)
sample=(targets.groupby('lr_size_bin',group_keys=False).apply(lambda g:g.sample(min(N_PER_BIN,len(g)),random_state=SEED)).reset_index(drop=True))
display(sample.groupby('lr_size_bin').size().to_frame('selected'))
manifest=OUT/'pilot_targets.csv'; sample.to_csv(manifest,index=False); print(manifest)


## 3. Instrument the actual Original DifIISR inference

The official pinned `inference.py` calls `sampler.inference(..., one_step=True)`, so the frozen baseline stops after the highest diffusion timestep and uses `pred_xstart` as the final latent sample.

The official sampler also reflect-pads each LR image to a multiple of 64 before encoding. This is required by the Swin blocks and is why we must reproduce that padding here. After decoding, we crop back to the original ×4 output size exactly like `sample_func`.


In [ ]:
script=OUT/'run_target_dynamics.py'
script.write_text(r'''import argparse,sys,random,math
from pathlib import Path
import numpy as np,pandas as pd,torch,torch.nn.functional as F
from PIL import Image
from omegaconf import OmegaConf
sys.path.insert(0,'/content/DifIISR')
from sampler import DifIISRSampler

ap=argparse.ArgumentParser(); ap.add_argument('--manifest'); ap.add_argument('--lr'); ap.add_argument('--out'); ap.add_argument('--config'); ap.add_argument('--seed',type=int,default=12345); a=ap.parse_args()
random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)
cfg=OmegaConf.load(a.config); cfg.model.ckpt_path='/content/DifIISR/weights/DifIISR.pth'; cfg.diffusion.params.sf=4; cfg.autoencoder.ckpt_path='/content/DifIISR/weights/autoencoder_vq_f4.pth'
if not Path(cfg.autoencoder.ckpt_path).exists():
 from basicsr.utils.download_util import load_file_from_url
 load_file_from_url(url='https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth',model_dir='/content/DifIISR/weights',progress=True,file_name='autoencoder_vq_f4.pth')
s=DifIISRSampler(cfg,chop_size=512,chop_stride=448,chop_bs=1,use_fp16=True,seed=a.seed,ddim=True)
df=pd.read_csv(a.manifest); lrdir=Path(a.lr); out=Path(a.out); out.mkdir(parents=True,exist_ok=True)

def metric(im,xc,yc,outer,core,noise_floor=1e-6):
 H,W=im.shape; cx=int(round(xc*W)); cy=int(round(yc*H)); ho=outer//2; hc=core//2; x0,x1=cx-ho,cx+ho+1; y0,y1=cy-ho,cy+ho+1
 if x0<0 or y0<0 or x1>W or y1>H:return None
 p=im[y0:y1,x0:x1].astype(np.float32); c0=ho-hc; c1=c0+core; c=p[c0:c1,c0:c1]; m=np.ones_like(p,dtype=bool); m[c0:c1,c0:c1]=False; ring=p[m]
 rs=float(ring.std()); scr=(float(c.max())-float(ring.mean()))/max(rs,noise_floor)
 x=np.maximum(0.,c.ravel()-float(ring.mean())); l1=float(np.abs(x).sum()); l2=float(np.linalg.norm(x)); n=x.size
 if l2<=0:h=0.
 else:
  raw=(np.sqrt(n)-l1/l2)/(np.sqrt(n)-1.); expected=np.sqrt(n/2.)*rs; excess=max(0.,l2-expected); h=float(raw*excess/(excess+expected+1e-6))
 return scr,h

rows=[]
for k,r in df.iterrows():
 ip=lrdir/r.image; arr=np.asarray(Image.open(ip).convert('RGB'),dtype=np.float32)/255.; y=torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).cuda()
 y=(y-.5)/.5
 ori_h,ori_w=y.shape[-2:]
 if ori_h>512 or ori_w>512: print('SKIP_CHOP',r.image); continue
 # Reproduce DifIISR.sample_func preprocessing: reflect-pad LR to multiples of 64.
 desired=64; pad_h=(math.ceil(ori_h/desired))*desired-ori_h; pad_w=(math.ceil(ori_w/desired))*desired-ori_w
 if pad_h or pad_w: y=F.pad(y,(0,pad_w,0,pad_h),mode='reflect')
 model_kwargs={'lq':y} if s.configs.model.params.cond_lq else None
 z_y=s.base_diffusion.encode_first_stage(y,s.autoencoder,up_sample=True); noise=torch.randn_like(z_y); z=s.base_diffusion.prior_sample(z_y,noise)
 i=s.base_diffusion.num_timesteps-1; t=torch.tensor([i],device=y.device)
 with torch.no_grad():
  o=s.base_diffusion.ddim_sample(model=s.model,x=z,y=z_y,t=t,clip_denoised=False,model_kwargs=model_kwargs)
  pred=s.base_diffusion.decode_first_stage(o['pred_xstart'],s.autoencoder).clamp(-1,1)
 pred=pred[:,:,:ori_h*4,:ori_w*4]
 gray=((pred[0].float().mean(0).cpu().numpy()+1.)*.5*255.).clip(0,255)
 mh=metric(gray,r.xc,r.yc,61,31)
 if mh is not None: rows.append(dict(image=r.image,target_idx=int(r.target_idx),lr_size_bin=r.lr_size_bin,max_dim_lr=r.max_dim_lr,step=0,timestep=i,scr=mh[0],hoyer=mh[1]))
 print(k+1,'/',len(df),r.image)
pd.DataFrame(rows).to_csv(out/'pilot_target_dynamics.csv',index=False)
print('DONE',len(rows))
''')
cmd=[PY,'-u',str(script),'--manifest',str(manifest),'--lr',str(LR_DIR),'--out',str(OUT),'--config',str(difiisr_dir/'configs'/'DifIISR_test.yaml'),'--seed',str(SEED)]
print('Ready to run:', ' '.join(map(str,cmd)))


## 4. Run the pilot


In [ ]:
env=os.environ.copy(); env['MPLBACKEND']='Agg'; env['CUDA_VISIBLE_DEVICES']='0'
result=subprocess.run(list(map(str,cmd)),cwd=str(difiisr_dir),env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout, flush=True)
if result.returncode!=0: raise RuntimeError(f'Target-dynamics pilot failed with exit code {result.returncode}. The real subprocess error is printed directly above.')
csv=OUT/'pilot_target_dynamics.csv'; assert csv.exists(); dynamics=pd.read_csv(csv); print('Rows:',len(dynamics)); display(dynamics.head())
